# Motor de Busqueda de Audios Similares

**DIEGO BESADA RODRÍGUEZ**

<div>
  <img src="images/music.png"
       alt="imagen música"
       width="500">
</div>

<br>

<div style="border:2px solid orange; padding:10px; border-radius:5px; background-color:#fff3cd;">
<strong>Estado:</strong> El entrenamiento de la red neuronal no ha ido como se esperaba. Los resultados no son satisfactorios.
</div>

<br>

Para esta práctica se propone desarrollar un motor de búsqueda de audios similares utilizando técnicas de Deep Learning. El objetivo es crear un sistema que, dado un audio de entrada, sea capaz de encontrar y mostrar audios similares dentro de una base de datos.

Para ello, la práctica sugiere utilizar espectrogramas como entrada de una red neuronal. Sin embargo, en este notebook se ha optado por un enfoque multimodal, incorporando además otras representaciones del audio como el espectrograma, el cromagrama y el tempograma.

Cada una de estas representaciones se procesa mediante una red convolucional independiente. Posteriormente, las características extraídas se concatenan y se introducen en una capa densa común.

La red se entrena inicialmente como un clasificador. Una vez entrenada, se elimina la última capa de clasificación para utilizar las salidas como embeddings de los audios.

Finalmente, estos embeddings se emplean para calcular la similitud entre audios y recuperar los más parecidos a una consulta dada.

In [1]:
import librosa
import matplotlib.pyplot as plt
import numpy as np
import random
import os
import gc
from tqdm.auto import tqdm

In [2]:
def generate_audio_representations(audio_file, output_root_dir, genre, clip_seconds=15, num_samples=10, rng_seed=42):
    '''
    Genera multiples muestras por cancion y guarda 4 imagenes por muestra:
    waveform, spectrogram, chromagram y tempogram.

    Salida forzada: output_root_dir/genre/{waveform,spectrogram,chromagram,tempogram}
    Imagenes limpias para DL: sin ejes, sin colorbar, sin margenes.
    '''

    target_height = 256
    target_width = 256

    def resize_2d(arr, height=target_height, width=target_width):
        # Fuerza mismo alto y ancho para todas las representaciones.
        if arr.ndim != 2:
            return arr
        h, w = arr.shape
        if h == height and w == width:
            return arr

        row_idx = np.linspace(0, h - 1, num=height)
        col_idx = np.linspace(0, w - 1, num=width)
        row_idx = np.clip(np.round(row_idx).astype(int), 0, h - 1)
        col_idx = np.clip(np.round(col_idx).astype(int), 0, w - 1)
        return arr[np.ix_(row_idx, col_idx)]

    def save_array_image(arr, path, cmap, vmin=None, vmax=None):
        plt.imsave(path, arr, cmap=cmap, vmin=vmin, vmax=vmax, origin="lower")

    # Obtener duracion sin cargar el audio completo
    audio_duration = librosa.get_duration(path=audio_file)
    if audio_duration <= 0:
        raise ValueError(f"Duracion invalida para {audio_file}")

    max_offset = max(0.0, audio_duration - clip_seconds)
    output_genre_dir = os.path.join(output_root_dir, genre)

    paths = {
        "waveform": os.path.join(output_genre_dir, "waveform"),
        "spectrogram": os.path.join(output_genre_dir, "spectrogram"),
        "chromagram": os.path.join(output_genre_dir, "chromagram"),
        "tempogram": os.path.join(output_genre_dir, "tempogram"),
    }

    for p in paths.values():
        os.makedirs(p, exist_ok=True)

    base_name = os.path.splitext(os.path.basename(audio_file))[0]
    rng = random.Random(f"{base_name}_{rng_seed}")

    generated_samples = 0
    skipped_samples = 0
    generated_paths = []
    skipped_paths = []

    sample_bar = tqdm(range(num_samples), desc=f"{base_name}", leave=False, position=1)

    for sample_idx in sample_bar:
        sample_suffix = f"s{sample_idx:02d}"
        sample_base = f"{base_name}_{sample_suffix}"

        sample_paths = {
            "waveform": os.path.join(paths["waveform"], f"{sample_base}_waveform.jpg"),
            "spectrogram": os.path.join(paths["spectrogram"], f"{sample_base}_spectrogram.jpg"),
            "chromagram": os.path.join(paths["chromagram"], f"{sample_base}_chromagram.jpg"),
            "tempogram": os.path.join(paths["tempogram"], f"{sample_base}_tempogram.jpg"),
        }

        if all(os.path.exists(path) for path in sample_paths.values()):
            skipped_samples += 1
            skipped_paths.append({"sample_idx": sample_idx, "paths": sample_paths})
            sample_bar.set_postfix(status="skipped")
            continue

        start_time = 0.0 if max_offset == 0 else rng.uniform(0, max_offset)
        y, sr = librosa.load(
            audio_file,
            sr=22050,
            mono=True,
            offset=start_time,
            duration=clip_seconds,
        )

        if y.size == 0:
            raise ValueError(f"No se pudo cargar audio util desde {audio_file} (muestra {sample_suffix})")

        # ===== Features =====
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr)
        mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
        chroma = librosa.feature.chroma_stft(y=y, sr=sr, tuning=False)
        tempogram = librosa.feature.tempogram(y=y, sr=sr)

        # Waveform en formato imagen 2D para mantener consistencia visual sin ejes.
        wave_norm = np.clip((y + 1.0) / 2.0, 0.0, 1.0)
        waveform_img = np.tile(wave_norm.reshape(1, -1), (128, 1))

        # Forzar mismo alto y ancho para todas las representaciones.
        waveform_img = resize_2d(waveform_img)
        mel_spec_db = resize_2d(mel_spec_db)
        chroma = resize_2d(chroma)
        tempogram = resize_2d(np.clip(tempogram, 0.0, 1.0))

        # RANGOS FIJOS (consistentes entre muestras)
        save_array_image(waveform_img, sample_paths["waveform"], cmap="magma", vmin=0.0, vmax=1.0)
        save_array_image(mel_spec_db, sample_paths["spectrogram"], cmap="magma", vmin=-80.0, vmax=0.0)
        save_array_image(chroma, sample_paths["chromagram"], cmap="coolwarm", vmin=0.0, vmax=1.0)
        save_array_image(tempogram, sample_paths["tempogram"], cmap="viridis", vmin=0.0, vmax=1.0)

        generated_samples += 1
        generated_paths.append({"sample_idx": sample_idx, "paths": sample_paths})
        sample_bar.set_postfix(status="done")

    sample_bar.close()

    return {
        "generated_samples": generated_samples,
        "skipped_samples": skipped_samples,
        "num_samples": num_samples,
        "generated_paths": generated_paths,
        "skipped_paths": skipped_paths,
    }

In [3]:
# PROCESADO COMPLETO: generar múltiples representaciones por canción con reanudación segura
root_directory = "Datasets/Datasets_audio/Music_Genres_Dataset"
output_directory = "Data/Music_Genres_Features_Dataset_ALL"
clip_seconds = 15
num_samples_per_song = 10

os.makedirs(output_directory, exist_ok=True)

# Recolectar todos los mp3 junto con su género
all_songs = []
for genre_folder in os.listdir(root_directory):
    genre_folder_path = os.path.join(root_directory, genre_folder)
    if os.path.isdir(genre_folder_path):
        for mp3_file in os.listdir(genre_folder_path):
            if mp3_file.endswith('.mp3'):
                mp3_file_path = os.path.join(genre_folder_path, mp3_file)
                all_songs.append((genre_folder, mp3_file_path))

total_songs = len(all_songs)
expected_images_total = total_songs * num_samples_per_song * 4
print(f"Canciones encontradas: {total_songs}")
print(f"Objetivo: {num_samples_per_song} muestras/canción de {clip_seconds}s -> {expected_images_total} imágenes")

ok_count = 0
error_count = 0
skipped_count = 0
incomplete_count = 0

for i, (genre, mp3_file_path) in enumerate(
    tqdm(all_songs, total=total_songs, desc="Procesando canciones", unit="song"),
    start=1,
):
    try:
        output_genre_folder = os.path.join(output_directory, genre)
        base_name = os.path.splitext(os.path.basename(mp3_file_path))[0]

        expected_paths = []
        for sample_idx in range(num_samples_per_song):
            sample_suffix = f"s{sample_idx:02d}"
            sample_base = f"{base_name}_{sample_suffix}"
            expected_paths.extend([
                os.path.join(output_genre_folder, "waveform", f"{sample_base}_waveform.jpg"),
                os.path.join(output_genre_folder, "spectrogram", f"{sample_base}_spectrogram.jpg"),
                os.path.join(output_genre_folder, "chromagram", f"{sample_base}_chromagram.jpg"),
                os.path.join(output_genre_folder, "tempogram", f"{sample_base}_tempogram.jpg"),
            ])

        # Si la canción ya está completa, se omite
        if all(os.path.exists(path) for path in expected_paths):
            skipped_count += 1
            continue

        result = generate_audio_representations(
            mp3_file_path,
            output_directory,
            genre,
            clip_seconds=clip_seconds,
            num_samples=num_samples_per_song,
            rng_seed=42,
        )

        # Verificación post-generación por canción
        missing_after = [path for path in expected_paths if not os.path.exists(path)]
        if len(missing_after) == 0:
            ok_count += 1
        else:
            incomplete_count += 1
            print(f"INCOMPLETA -> {mp3_file_path} (faltan {len(missing_after)} imágenes)")

    except Exception as e:
        error_count += 1
        print(f"ERROR -> {mp3_file_path}")
        print(f"Motivo: {e}")
    finally:
        if i % 25 == 0:
            gc.collect()

# Conteo final de imágenes realmente generadas
actual_images_total = 0
for root, _, files in os.walk(output_directory):
    actual_images_total += sum(1 for f in files if f.endswith('.jpg'))

print("\nResumen final")
print(f"Canciones completas generadas en esta ejecución: {ok_count}")
print(f"Canciones ya completas y omitidas: {skipped_count}")
print(f"Canciones incompletas tras intentar generar: {incomplete_count}")
print(f"Canciones con error: {error_count}")
print(f"Imágenes esperadas totales: {expected_images_total}")
print(f"Imágenes actuales en disco: {actual_images_total}")
print(f"Salida en: {output_directory}")

Canciones encontradas: 983
Objetivo: 10 muestras/canción de 15s -> 39320 imágenes


Procesando canciones:   0%|          | 0/983 [00:00<?, ?song/s]


Resumen final
Canciones completas generadas en esta ejecución: 0
Canciones ya completas y omitidas: 983
Canciones incompletas tras intentar generar: 0
Canciones con error: 0
Imágenes esperadas totales: 39320
Imágenes actuales en disco: 39320
Salida en: Data/Music_Genres_Features_Dataset_ALL


## 1. Preparar Dataset

In [4]:
import copy
import time
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

In [5]:
class AudioDataset(Dataset):
    def __init__(self, root_dir, mean=None, std=None):
        self.root_dir = root_dir
        self.representations = ["waveform", "spectrogram", "chromagram", "tempogram"]
        self.classes = sorted(
            [name for name in os.listdir(root_dir) if os.path.isdir(os.path.join(root_dir, name))]
        )
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        self.data = self.load_data()
        self.transform = transforms.Compose([transforms.Resize((256, 256)), transforms.ToTensor()])
        
        # Usar estadísticas globales del dataset en lugar de por-imagen
        if mean is not None and std is not None:
            self.mean = torch.tensor(mean).view(3, 1, 1)
            self.std = torch.tensor(std).view(3, 1, 1)
        else:
            # Valores por defecto (ImageNet) si no se proporcionan
            self.mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
            self.std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def load_data(self):
        data = []
        for class_name in self.classes:
            class_dir = os.path.join(self.root_dir, class_name)
            rep_dirs = {rep: os.path.join(class_dir, rep) for rep in self.representations}

            if not all(os.path.isdir(rep_dir) for rep_dir in rep_dirs.values()):
                continue

            waveform_files = [
                f for f in os.listdir(rep_dirs["waveform"]) if f.endswith(".jpg")
            ]

            for filename in waveform_files:
                base_name = filename.replace("_waveform.jpg", "")
                paths = {
                    rep: os.path.join(rep_dirs[rep], f"{base_name}_{rep}.jpg")
                    for rep in self.representations
                }

                if all(os.path.exists(path) for path in paths.values()):
                    label = self.class_to_idx[class_name]
                    data.append((paths, label))

        return data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        paths, label = self.data[idx]
        images = []

        for rep in self.representations:
            img = Image.open(paths[rep]).convert("RGB")
            img = self.transform(img)
            # Usar normalización global del dataset, NO por-imagen
            img = (img - self.mean) / (self.std + 1e-7)
            images.append(img)

        return tuple(images), label

In [6]:
def compute_dataset_statistics_from_indices(dataset, indices):
    """
    Calcula media y std SOLO con muestras de train para evitar fuga de datos.
    """
    sums = torch.zeros(3)
    sum_of_squares = torch.zeros(3)
    num_images = 0

    print("Calculando estadísticas del TRAIN split...")

    for idx in tqdm(indices, desc="Stats train", leave=False):
        paths, _ = dataset.data[idx]

        for rep in dataset.representations:
            try:
                img = Image.open(paths[rep]).convert("RGB")
                img_tensor = dataset.transform(img)  # Resize + ToTensor (sin normalizar)
                sums += img_tensor.mean(dim=[1, 2])
                sum_of_squares += (img_tensor ** 2).mean(dim=[1, 2])
                num_images += 1
            except Exception as e:
                print(f"Error procesando {paths[rep]}: {e}")

    mean = sums / num_images
    std = torch.sqrt(sum_of_squares / num_images - mean ** 2)

    print(f"Train stats (N={num_images} imágenes):")
    print(f"  Mean: {mean.tolist()}")
    print(f"  Std:  {std.tolist()}")

    return mean.tolist(), std.tolist()

In [7]:
import re


def get_song_id_from_waveform_path(waveform_path: str) -> str:
    """Extrae id de canción ignorando el sufijo de muestra _sXX."""
    sample_name = os.path.basename(waveform_path).replace("_waveform.jpg", "")
    return re.sub(r"_s\d{2}$", "", sample_name)


root_dir_dataset = "Data/Music_Genres_Features_Dataset_ALL"
batch_size = 32
seed = 42
train_ratio = 0.8

# 1) Dataset base (sin stats calculadas aún)
dataset = AudioDataset(root_dir=root_dir_dataset)

# 2) Agrupar índices por canción para que todas sus muestras queden en el mismo split
song_to_indices = {}
for i, (paths, _) in enumerate(dataset.data):
    song_id = get_song_id_from_waveform_path(paths["waveform"])
    song_to_indices.setdefault(song_id, []).append(i)

all_songs_ids = list(song_to_indices.keys())
rng = random.Random(seed)
rng.shuffle(all_songs_ids)

train_song_count = int(len(all_songs_ids) * train_ratio)
train_songs = set(all_songs_ids[:train_song_count])
valid_songs = set(all_songs_ids[train_song_count:])

train_indices = [idx for s in train_songs for idx in song_to_indices[s]]
valid_indices = [idx for s in valid_songs for idx in song_to_indices[s]]

# 3) Verificación
shared_songs = train_songs.intersection(valid_songs)
print(f"Canciones compartidas train-valid: {len(shared_songs)}")

# 4) Calcular stats SOLO con train y aplicarlas al dataset completo
dataset_mean, dataset_std = compute_dataset_statistics_from_indices(dataset, train_indices)
dataset.mean = torch.tensor(dataset_mean).view(3, 1, 1)
dataset.std = torch.tensor(dataset_std).view(3, 1, 1)

# 5) Subsets y dataloaders
train_dataset = torch.utils.data.Subset(dataset, train_indices)
test_dataset = torch.utils.data.Subset(dataset, valid_indices)

train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print(f"Train samples: {len(train_dataset)}")
print(f"Valid samples: {len(test_dataset)}")
print(f"Canciones train: {len(train_songs)}")
print(f"Canciones valid: {len(valid_songs)}")

data_loader = {}
data_loader["train"] = train_dataloader
data_loader["valid"] = test_dataloader

Canciones compartidas train-valid: 0
Calculando estadísticas del TRAIN split...


Stats train:   0%|          | 0/7870 [00:00<?, ?it/s]

Train stats (N=31480 imágenes):
  Mean: [0.5588973760604858, 0.3086003065109253, 0.5241169333457947]
  Std:  [0.2524634301662445, 0.23537079989910126, 0.19987402856349945]
Train samples: 7870
Valid samples: 1960
Canciones train: 768
Canciones valid: 193


## Creación red neuronal

In [8]:
def train_model(
    model,
    criterion,
    optimizer,
    scheduler,
    data_loader,
    n_epochs=10,
    device=None,
    early_stopping_patience=6,
    min_delta=1e-4,
 ):
    if device is None:
        if torch.backends.mps.is_available():
            device = torch.device("mps")
        elif torch.cuda.is_available():
            device = torch.device("cuda")
        else:
            device = torch.device("cpu")

    model = model.to(device)

    # keeping track of best weights
    best_model_wts = copy.deepcopy(model.state_dict())
    valid_loss_min = np.inf
    epochs_without_improve = 0

    for epoch in range(1, n_epochs + 1):
        since = time.time()
        running_losses = {}
        running_accuracies = {}

        for phase in ["train", "valid"]:
            running_loss = 0.0
            running_acc = 0.0

            if phase == "train":
                model.train()
            else:
                model.eval()

            phase_bar = tqdm(
                data_loader[phase],
                desc=f"Epoch {epoch}/{n_epochs} [{phase}]",
                leave=False
            )

            for data, target in phase_bar:
                target = target.to(device)

                # Orden del dataset: waveform, spectrogram, chromagram, tempogram
                if isinstance(data, (list, tuple)):
                    wave, spec, chroma, temp = [x.to(device) for x in data]
                    inputs = (spec, temp, chroma, wave)
                else:
                    data = data.to(device)
                    inputs = (data,)

                if phase == "train":
                    optimizer.zero_grad()

                with torch.set_grad_enabled(phase == "train"):
                    output = model(*inputs)
                    loss = criterion(output, target)

                    if phase == "train":
                        loss.backward()
                        optimizer.step()

                _, preds = torch.max(output, 1)
                running_acc += torch.sum(preds == target).item()
                running_loss += loss.item() * target.size(0)

                phase_bar.set_postfix(loss=f"{loss.item():.4f}")

            running_losses[phase] = running_loss / len(data_loader[phase].dataset)
            running_accuracies[phase] = running_acc

            if phase == "valid" and scheduler is not None:
                if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
                    scheduler.step(running_losses[phase])
                else:
                    scheduler.step()

        train_accuracy = running_accuracies["train"] / len(data_loader["train"].dataset)
        valid_accuracy = running_accuracies["valid"] / len(data_loader["valid"].dataset)

        print(
            "Epoch: {} 	Training Loss: {:.6f} 	Validation Loss: {:.6f} 	Training Accuracy: {:.6f} ({}/{}) 	Validation Accuracy: {:.6f} ({}/{})".format(
                epoch,
                running_losses["train"],
                running_losses["valid"],
                train_accuracy,
                int(running_accuracies["train"]),
                len(data_loader["train"].dataset),
                valid_accuracy,
                int(running_accuracies["valid"]),
                len(data_loader["valid"].dataset),
            )
        )

        # save model if validation loss has decreased enough
        if running_losses["valid"] < (valid_loss_min - min_delta):
            print(
                "Validation loss decreased ({:.6f} --> {:.6f}). Saving model ...".format(
                    valid_loss_min, running_losses["valid"]
                )
            )
            best_model_wts = copy.deepcopy(model.state_dict())
            valid_loss_min = running_losses["valid"]
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1
            print(f"No mejora en validación ({epochs_without_improve}/{early_stopping_patience})")

        elapsed_time = time.time() - since
        print()
        print("Epoch completed in {:.0f}m {:.0f}s".format(elapsed_time // 60, elapsed_time % 60))

        if device.type == "cuda":
            torch.cuda.empty_cache()

        if epochs_without_improve >= early_stopping_patience:
            print("Early stopping activado.")
            break

    model.load_state_dict(best_model_wts)
    return model

In [9]:
def save_checkpoint(model, optimizer, scheduler, model_file):
    parameters = {
        'state_dict': model.state_dict(),
        'optimizer': optimizer,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler': scheduler,
        'scheduler_state_dict': scheduler.state_dict()
    }
    torch.save(parameters, model_file)

In [10]:
import torch.nn as nn
from torchvision import models

In [11]:
class ResNetBranch(nn.Module):
    def __init__(self, out_dim=64, pretrained=True, train_backbone=False):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        backbone = models.resnet18(weights=weights)
        num_features = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone

        # Por defecto congelamos todo el backbone y solo entrenamos la proyección.
        if not train_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
        else:
            for param in self.backbone.parameters():
                param.requires_grad = False
            for param in self.backbone.layer4.parameters():
                param.requires_grad = True

        self.proj = nn.Sequential(
            nn.Linear(num_features, out_dim),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(out_dim),
            nn.Dropout(0.45),
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.proj(x)
        return x


class MultiModalAudioClassifier(nn.Module):
    def __init__(self, num_classes, branch_dim=64, pretrained=True, train_backbone=False):
        super().__init__()
        self.spec_branch = ResNetBranch(out_dim=branch_dim, pretrained=pretrained, train_backbone=train_backbone)
        self.temp_branch = ResNetBranch(out_dim=branch_dim, pretrained=pretrained, train_backbone=train_backbone)
        self.chroma_branch = ResNetBranch(out_dim=branch_dim, pretrained=pretrained, train_backbone=train_backbone)
        self.wave_branch = ResNetBranch(out_dim=branch_dim, pretrained=pretrained, train_backbone=train_backbone)

        # Cabeza más pequeña para reducir la capacidad de memorizar el train.
        self.classifier = nn.Sequential(
            nn.Linear(branch_dim * 4, 128),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(128),
            nn.Dropout(0.50),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(64),
            nn.Dropout(0.35),
            nn.Linear(64, num_classes),
        )

    def encode(self, spec, temp, chroma, wave):
        """Devuelve el embedding fusionado (antes de la capa de clasificación)."""
        f_spec = self.spec_branch(spec)
        f_temp = self.temp_branch(temp)
        f_chroma = self.chroma_branch(chroma)
        f_wave = self.wave_branch(wave)
        fused = torch.cat([f_spec, f_temp, f_chroma, f_wave], dim=1)
        return fused

    def forward(self, spec, temp, chroma, wave):
        fused = self.encode(spec, temp, chroma, wave)
        return self.classifier(fused)

num_classes = len(dataset.classes)
model = MultiModalAudioClassifier(num_classes=num_classes, pretrained=True, train_backbone=False)
print(f"Modelo multimodal con backbone congelado y cabeza más pequeña creado con {num_classes} clases.")

Modelo multimodal con backbone congelado y cabeza más pequeña creado con 10 clases.


In [12]:
learning_rate = 1e-4
backbone_learning_rate = 1e-5
training_epochs = 50
model_name = "model_audio_multimodal_v4"
checkpoint_path = model_name + ".pt"
train_before_saving = True
load_existing_checkpoint = True  # Experimento limpio
just_load = True  # Solo cargar pesos sin entrenar ni guardar

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if just_load:
    model = model
    
else: 
    print(f"Dispositivo de entrenamiento: {device}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.10)

    head_params = []
    backbone_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        if "backbone.layer4" in name:
            backbone_params.append(param)
        else:
            head_params.append(param)

    optimizer = torch.optim.AdamW(
        [
            {"params": head_params, "lr": learning_rate},
            {"params": backbone_params, "lr": backbone_learning_rate},
        ],
        weight_decay=5e-3,
    )
    m_lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=2, factor=0.5
    )

    if load_existing_checkpoint and os.path.exists(checkpoint_path):
        print(f"Cargando pesos previos desde {checkpoint_path}...")
        checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        state_dict = checkpoint["state_dict"] if isinstance(checkpoint, dict) and "state_dict" in checkpoint else checkpoint
        missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
        if missing_keys:
            print(f"Capas no encontradas en checkpoint: {len(missing_keys)}")
        if unexpected_keys:
            print(f"Capas extra en checkpoint: {len(unexpected_keys)}")
    else:
        print("Entrenamiento limpio desde inicializacion (sin checkpoint previo).")

    if train_before_saving:
        model = train_model(
            model,
            criterion,
            optimizer,
            m_lr_scheduler,
            data_loader,
            n_epochs=training_epochs,
            device=device,
            early_stopping_patience=50,
        )

        print("Saving best model...")
        save_checkpoint(model, optimizer, m_lr_scheduler, checkpoint_path)
    else:
        print("train_before_saving=False -> No se entrena ni se sobrescribe el checkpoint.")

Durante el desarrollo de este proyecto, el modelo experimentó un overfitting severo que limitó significativamente el rendimiento de la red neuronal. A pesar de múltiples intentos de mejora implementando diferentes técnicas de regularización, el problema persistió:

- **Síntomas observados**: La pérdida de validación se estancaba mientras que la pérdida de entrenamiento continuaba disminuyendo, indicando que el modelo memorizaba el conjunto de entrenamiento sin generalizar adecuadamente.
- **Estrategias intentadas**: Se implementaron varias técnicas de regularización incluyendo:
  - Frozen backbone con congelación selectiva de capas
  - Dropout agresivo (0.35 a 0.50)
  - Batch normalization
  - Label smoothing (0.10)
  - Weight decay (5e-3)
  - Arquitectura más compacta en la cabeza de clasificación
  - Early stopping con paciencia aumentada
  - Genera más datos (en lugar de una muestra por audio, generar 10)

Sin embargo, ninguna de estas mejoras logró resolver el problema.

Es posible que el enfoque multimodal resultase demasiado ambicioso para los datos y objetivos del proyecto y que una arquitectura más simple (solo espectrogramas) fuera haber sido más efectiva.

A continuación se continua el desarrollo del proyecto, aunque, dada las capacidades de la red, los resultados del buscador serán (más que) limitados.

## Extracción de embeddings

In [13]:
import joblib
import pandas as pd
import torch.nn.functional as F
import re


def get_song_id_from_waveform_path(waveform_path: str) -> str:
    sample_name = os.path.basename(waveform_path).replace("_waveform.jpg", "")
    return re.sub(r"_s\d{2}$", "", sample_name)


@torch.no_grad()
def extract_embedding(model, spec, temp, chroma, wave):
    """Devuelve embedding L2-normalizado de una muestra multimodal."""
    model.eval()
    if hasattr(model, "encode"):
        emb = model.encode(spec, temp, chroma, wave)
    else:
        f_spec = model.spec_branch(spec)
        f_temp = model.temp_branch(temp)
        f_chroma = model.chroma_branch(chroma)
        f_wave = model.wave_branch(wave)
        emb = torch.cat([f_spec, f_temp, f_chroma, f_wave], dim=1)
    emb = F.normalize(emb, p=2, dim=1)
    return emb


def load_multimodal_tensors(paths, dataset):
    """Carga una muestra (4 imágenes) y aplica la misma normalización del training."""
    images = []
    for rep in dataset.representations:
        img = Image.open(paths[rep]).convert("RGB")
        img = dataset.transform(img)
        img = (img - dataset.mean) / (dataset.std + 1e-7)
        images.append(img)

    wave, spec, chroma, temp = images
    # Orden esperado por el modelo: spec, temp, chroma, wave
    return (
        spec.unsqueeze(0),
        temp.unsqueeze(0),
        chroma.unsqueeze(0),
        wave.unsqueeze(0),
    )


@torch.no_grad()
def extract_embedding_from_paths(model, dataset, device, paths):
    spec, temp, chroma, wave = load_multimodal_tensors(paths, dataset)
    spec = spec.to(device)
    temp = temp.to(device)
    chroma = chroma.to(device)
    wave = wave.to(device)
    emb = extract_embedding(model, spec, temp, chroma, wave)
    return emb.squeeze(0).cpu().numpy().astype(np.float32)


def build_audio_embedding_index(
    model,
    dataset,
    device,
    index_joblib_path="data/audio_search_index.joblib",
    index_csv_path="data/audio_search_index.csv",
):
    """Extrae embeddings de toda la base y los guarda para búsqueda."""
    model = model.to(device)
    model.eval()

    vectors = []
    metadata = []
    idx_to_class = {v: k for k, v in dataset.class_to_idx.items()}

    for i, (paths, label) in enumerate(tqdm(dataset.data, desc="Extrayendo embeddings")):
        emb_np = extract_embedding_from_paths(model, dataset, device, paths)
        vectors.append(emb_np)

        sample_name = os.path.basename(paths["waveform"]).replace("_waveform.jpg", "")
        song_id = get_song_id_from_waveform_path(paths["waveform"])
        metadata.append(
            {
                "sample_id": i,
                "genre": idx_to_class[label],
                "song_id": song_id,
                "sample_name": sample_name,
                "waveform_path": paths["waveform"],
                "spectrogram_path": paths["spectrogram"],
                "chromagram_path": paths["chromagram"],
                "tempogram_path": paths["tempogram"],
            }
        )

    embeddings = np.vstack(vectors).astype(np.float32)
    metadata_df = pd.DataFrame(metadata)

    os.makedirs(os.path.dirname(index_joblib_path), exist_ok=True)
    db = {
        "paths": metadata_df["waveform_path"].tolist(),
        "labels": metadata_df["genre"].tolist(),
        "features": embeddings,
        "metadata": metadata,
        "embedding_dim": embeddings.shape[1],
        "num_samples": embeddings.shape[0],
    }
    joblib.dump(db, index_joblib_path)
    metadata_df.to_csv(index_csv_path, index=False)

    print(f"Embeddings guardados en: {index_joblib_path}")
    print(f"Índice CSV guardado en: {index_csv_path}")
    print(f"Shape embeddings: {embeddings.shape}")

    return embeddings, metadata_df

In [14]:
embeddings, metadata_df = build_audio_embedding_index(model, dataset, device)
print("Primeras filas del índice de búsqueda:")
print(metadata_df.head())

Extrayendo embeddings:   0%|          | 0/9830 [00:00<?, ?it/s]

Embeddings guardados en: data/audio_search_index.joblib
Índice CSV guardado en: data/audio_search_index.csv
Shape embeddings: (9830, 256)
Primeras filas del índice de búsqueda:
   sample_id  genre                                            song_id  \
0          0  blues       Stay Around A Little Longer (Official Video)   
1          1  blues                           B.B. King - Ghetto woman   
2          2  blues              Stevie Ray Vaughan - Life by the drop   
3          3  blues  Little Walter - Blue And Lonesome [Digitally R...   
4          4  blues                      Etta James - At Last - Lyrics   

                                         sample_name  \
0   Stay Around A Little Longer (Official Video)_s04   
1                       B.B. King - Ghetto woman_s00   
2          Stevie Ray Vaughan - Life by the drop_s09   
3  Little Walter - Blue And Lonesome [Digitally R...   
4                  Etta James - At Last - Lyrics_s04   

                                       wa

In [15]:
search_index = {
    row.waveform_path: embeddings[i]
    for i, row in metadata_df.reset_index(drop=True).iterrows()
}

## 3. Buscador (usando utilities/searcher.py)

In [16]:
from utilities.searcher import Searcher
searcher = Searcher(search_index, metric="cosine")

In [17]:
query_row = metadata_df.sample(1, random_state=42).iloc[0]
query_paths = {
    "waveform": query_row["waveform_path"],
    "spectrogram": query_row["spectrogram_path"],
    "chromagram": query_row["chromagram_path"],
    "tempogram": query_row["tempogram_path"],
}
query_embedding = extract_embedding_from_paths(model, dataset, device, query_paths)

print("Consulta:")
print(f"  song_id: {query_row['song_id']}")
print(f"  sample_name: {query_row['sample_name']}")
print(f"  genre: {query_row['genre']}")

Consulta:
  song_id: Never Say Never
  sample_name: Never Say Never_s02
  genre: pop


In [18]:
ranked = searcher.search(query_embedding, top_k=50)

path_to_row = {row.waveform_path: row for _, row in metadata_df.iterrows()}
results = []
for rank, (distance, path) in enumerate(ranked, start=1):
    row = path_to_row[path]
    results.append(
        {
            "rank_raw": rank,
            "distance": float(distance),
            "similarity": float(1.0 - distance),
            "genre": row["genre"],
            "song_id": row["song_id"],
            "sample_name": row["sample_name"],
            "waveform_path": row["waveform_path"],
            "is_query_song": row["song_id"] == query_row["song_id"],
        }
    )

results_df = pd.DataFrame(results)

# Quitar todas las muestras de la misma canción query para medir recuperación real
results_no_self = results_df[~results_df["is_query_song"]].reset_index(drop=True)
results_no_self.insert(0, "rank", np.arange(1, len(results_no_self) + 1))

query_genre = query_row["genre"]
same_genre_at_5 = (results_no_self.head(5)["genre"] == query_genre).mean()
same_genre_at_10 = (results_no_self.head(10)["genre"] == query_genre).mean()

print(f"Query genre: {query_genre}")
print(f"same_genre@5:  {same_genre_at_5:.2f}")
print(f"same_genre@10: {same_genre_at_10:.2f}")

results_no_self[["rank", "distance", "similarity", "genre", "song_id", "sample_name"]].head(20)

Query genre: pop
same_genre@5:  0.20
same_genre@10: 0.20


,rank,distance,similarity,genre,song_id,sample_name
0,1,0.025118,0.974882,hiphop,Dj Format Feat Chali 2NA & Akil-We Know Somet...,Dj Format Feat Chali 2NA & Akil-We Know Somet...
1,2,0.025867,0.974133,disco,Jump Around,Jump Around_s08
2,3,0.025965,0.974035,pop,Lean On,Lean On_s00
3,4,0.025980,0.974020,disco,Party Like A Rock Star Closed Captioned,Party Like A Rock Star Closed Captioned_s05
4,5,0.026840,0.973160,hiphop,Big Shit Poppin' (Do It),Big Shit Poppin' (Do It)_s01
5,6,0.027129,0.972871,pop,Bad Romance,Bad Romance_s09
6,7,0.027712,0.972288,hiphop,I Luv It,I Luv It_s08
7,8,0.028442,0.971558,hiphop,Dj Format Feat Chali 2NA & Akil-We Know Somet...,Dj Format Feat Chali 2NA & Akil-We Know Somet...
8,9,0.028592,0.971408,disco,Daddy Yankee Rompe HQ,Daddy Yankee Rompe HQ_s02
9,10,0.028724,0.971276,hiphop,Stephen & Damian Marley ft Snoop Dogg - The T...,Stephen & Damian Marley ft Snoop Dogg - The T...


Como se ha mencionado, dada la calidad de los embeddings obtenidos, el buscador no ofrecerá resultados satisfactorios. Se puede ver como solo un 20% de los audios recuperados pertenecen a la misma clase que el audio de consulta, lo cual es un resultado muy pobre.